In [14]:

import requests

class Meijer:
    BASE_URL = "https://api.meijer.com"
    AUTH_URL = "https://id.meijer.com/oauth2/default/v1"

    def __init__(self, access_token="": str, refresh_token="": str, store_id: int = 314):
        self.access_token = access_token
        self.refresh_token = refresh_token
        self.store_id = store_id
        self.session = requests.Session()
        self.session.headers.update({
            "ocp-apim-subscription-key": "a10bc58ac484478d9b3958b1742c3a03",
            "user-agent": "Meijer/100900000 okhttp/4.12.0 Dalvik/2.1.0 (Linux; U; Android 10; One Build/QQ3A.200705.002)",
            "cookie": "ROUTE=.api-d7fbffc4d-9bg89"
        })
    
    
    def check_phone_conflict(self, phone_number: str):
        url = f"{self.BASE_URL}/loyalty/accounts/getConflictTypeByPhone/{phone_number}"
        headers = {
            "accept": "application/vnd.meijer.account.phoneNumberConflictStatus-v1.0+json",
            "content-type": "application/json"
        }
        response = self.session.get(url, headers=headers)
        return response.json() if response.ok else response.text

    def validate_phone_number(self, phone_number: str):
        url = f"{self.BASE_URL}/Loyalty/AccountLinking/validatephone"
        headers = {
            "accept": "application/json",
            "content-type": "application/json"
        }
        payload = {"PhoneNumber": phone_number}
        response = self.session.post(url, headers=headers, json=payload)
        return response.json() if response.ok else response.text


    def create_account_with_mperks(self, first_name: str, last_name: str, zip_code: str, store_id: int, email: str, password: str, phone_number: str, pin: str, enroll_in_pharmacy: bool = True):
        url = f"{self.BASE_URL}/loyalty/accounts/createAccountWithMperks"
        headers = {
            "accept": "application/vnd.meijer.account.updateConfirmation-v1.0+json",
            "content-type": "application/vnd.meijer.account.account-v1.0+json"
        }
        payload = {
            "firstName": first_name,
            "lastName": last_name,
            "zip": zip_code,
            "storeId": store_id,
            "email": email,
            "password": password,
            "phoneNumber": phone_number,
            "pin": pin,
            "enrollInPharmacy": enroll_in_pharmacy
        }
        response = self.session.post(url, headers=headers, json=payload)
        return response.json() if response.ok else response.text


In [26]:
meijer = Meijer("", "")

In [27]:
meijer.check_phone_conflict("3093775090")

{'conflictType': 2, 'name': 'DigitalConflict'}

In [28]:
meijer.validate_phone_number("5")

{'isValid': 'false',
 'isTextable': 'false',
 'deviceType': '',
 'description': '',
 'errorMsg': ''}

In [29]:
import random
from faker import Faker

In [30]:
# List of common Michigan area codes
michigan_area_codes = ["248", "313", "517", "586", "616", "734", "810", "906", "947"]

# Generate a fake phone number with a Michigan area code
area_code = random.choice(michigan_area_codes)
phone_number = f"{area_code}{random.randint(100,999)}{random.randint(1000,9999)}"  # Ensures a 10-digit number

fake = Faker()

In [31]:
fake = Faker()
payload = {
    "first_name": fake.first_name(),
    "last_name": fake.last_name(),
    "zip_code": "49441",
    "store_id": 19,
    "email": fake.email(),
    "password": "Default1",
    "phone_number": phone_number,
    "pin": "1234",
    "enroll_in_pharmacy": True
}

result = meijer.create_account_with_mperks(**payload)


In [32]:
result

{'id': 0, 'accountId': 26294950, 'success': True}

In [33]:
payload

{'first_name': 'Jason',
 'last_name': 'Burgess',
 'zip_code': '49441',
 'store_id': 19,
 'email': 'calvarez@example.com',
 'password': 'Default1',
 'phone_number': '5179918616',
 'pin': '1234',
 'enroll_in_pharmacy': True}

In [44]:
import requests

def identify_user(identifier: str, stateHandle: str):
    url = "https://id.meijer.com/idp/idx/identify"
    headers = {
        "accept": "application/json; okta-version=1.0.0",
        "x-okta-user-agent-extended": "okta-auth-js/7.11.0 okta-signin-widget-g3-7.30.0-g8c04eaf",
        "user-agent": "Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.0.0 Mobile Safari/537.36",
        "content-type": "application/json",
        "origin": "https://id.meijer.com",
        "x-requested-with": "com.duckduckgo.mobile.android",
        "sec-fetch-site": "same-origin",
        "sec-fetch-mode": "cors",
        "sec-fetch-dest": "empty",
        "accept-encoding": "gzip, deflate",
        "accept-language": "en-US,en;q=0.9"
    }
    
    payload = {
        "identifier": identifier,
        "stateHandle": stateHandle
    }

    response = requests.post(url, headers=headers, json=payload)
    return response

In [45]:
request = {
    "identifier": "calvarez@example.com",
    "stateHandle": "eyJ6aXAiOiJERUYiLCJhbGlhcyI6ImVuY3J5cHRpb25rZXkiLCJ2ZXIiOiIxIiwib2lkIjoiMDBvZW14bng4Q3NocE5WeVM2OTYiLCJlbmMiOiJBMjU2R0NNIiwiYWxnIjoiZGlyIn0..9ZU81bJzlvA22Ebc.-3tH7fir6SxVNFgulaJdaf_qGu5betHIJ7L9xyASykSB8iYUKosM8AyHFb2EYOFjsqK2-OpHdQVi9mFDYaymbUGhlV2-W9NW2qya2gpUUOdCtQTwgf9IYdKaT061EvqMNqxx_WvDsdq8nWxujby4fmwE5B_2k3SsVz9QEiDXZfLlTagiKKWogDeyZEO5QoL3DA0U0VKtbDrTROuol1Pbt2c6PI_zRz3aIvSP0YdwUDHaHj3V6uDszMdi0D_TEibcK1POHpTg-vdngSGi2QKEA6f4KN3C7UydBRtNjzRed_7_2P3YoBooQzT_Pem1iTPtAhAAV80YysLm34W18t07ZVUiy10tR5d_TuiorNu3gyuFIZYmyKXeD8xVT0ntVeCrVbyzm1OzRxD8F6QOh5fHmFO1tR6E27A0adDmQ5zEjzP3vMZH_H_I4szBoWV5jGQd82YmGNcBFgdrf96GEUSrVM_3nKhii13EQJzrlZAUu3jgU7lcfkDYIknM4iINssMICfyhfEfC7YlfoltfWQVBbaawyxbdivWNvQsnrqH74M5IrAH-w5_ngXxsHJp_5msyoQrD42AF9SKDqFROKmRIJBfFwLZ3uHXcQCDbGEBURU7T4nBG_AQAH59-nc5Ai3jvNxKWU2fjUm1_4h3_dMnp9yPoCRVg4v0cRTLolwvNYkInBx8PNrN4-_PoRBLvh9mNgIcHcWEwqPRK_LOHhaTQnpluP9vd-pHr6HATMZ7MRiwBParGN0LBSJ9UIOXCZDyQ7xTni3fPi4d1ItxO30TqfQdzAOlLCxJ3DhoWuTzlUp-xvPbSpezJNIB_4JIzxLU1VvR_oEyVTUZrHubswe9QkaZplaPZwpt9ho84xMkeVdpwVPBKdby7xE0-eTSiNStG4SkWD3lu1J94vXmdSaRErcSEprq8Hbk6vAU1FyrK-LwPnaQsno-Rd-jbZ_FwWqw3Q79o6T4MrVdFtPkiTQGY51HpJiuLD0NsyfJKEl_FKiFDILS5qRdZ68BOx69b2ORLi1CC2fBc5ZOlkhkYCioWfLgnL4-vGfxPlXQ72iu81xC39x18HfKsaD7gePVxt7ktjpOq0TzUZjCqRUMzT4CgxBfWAWg2yuoPcjysR7kNMgoG0cO-nQwEg35H28X2owIFGLi02w_SbQ31s8r8W39OaQD4E7JtJE3unPKD4SCCzbORQ2qrGZIth_XLX37e-EtpvZz81xbdeEZ2VVyTqtHUiO3AHZ15sI7wc1VFiXYP_gXTRHiIO9pelWi3__Dyr_Dw3lKzLLqHr88qFpjXNdHZ6Pc2ofxrjImjaYdIdaOHtFxnyDRntTPbBdHbwQSZtBVO-a5RDIW2wPPPbRzlBPBwLwsD6F9O8RG-PGeJBh7KQJxHC-tB8qp0aJuEwI7r2VCt4bIS9_MRrzaW-Cl8ymjWWOwMcQdzJgE706pjxAozdQSaoMjv7FAAfGtNI8db3obFV8ZeUcDqEU-k2uXqNdS4qMubEjat-VJ8InfCoOpv0I7duRlvdzVQJsWAezHXUjkuD6-1-JYXZkWGdun20OM09ZGzCNm-yDqkalfRY3yaEiySG5qZT2jUzPCAMIuDDTxdXguJaRFPOQ0G6Fv8je-_42Uj9NasRMbmxt4ZqhKtw17OPgPtgw1souqD1-GDIlpfi7vkuSZxIf_5rFpisTjnPxswSevCDShytrxbXnM-IgJq54dryL31T5ksDceYy-anqjUsQk6RZbkyxUYzkhTXvClpnx7JXSzkmOuv5vXsPdXT2F7rVbjvyWb_pLAHGzJsd0ua6k06a6PcBPQEnisff_NODwnyEa0aZGXUVVp9z6OySIWamLFx3wHbOlgiPx3grHfnD6-NIYqsUvuRQ4tagbuZvzQhdy_tCFCFzuUZScBwu8wFY230tWPK0Zd0wX0_F2R5zbp-MTKzq06bK30VrYMuxS6LbTvOsPfdw4ssHTllF59qbcEr_9GJ-jNLvONVqLcbewLeR19bfznlGqwepGvJxCfIgmvnA7uPikbvP8LW6ULmFtRi8u7CsMJNTPvpxGwMMr95-BtRpEM5nJHVjO-4QQ0Ogl7rs_2IpRcWkUSRM3nc_qKdkPbmm7lqfIRCi_zAfS4fetyszTaVeU7tKs1BB7YDgi0H2o1SVPdV0Ef7sXvrPVsVZD_2nUz8vFkeVmUMawYiYB6kcdQY72OnfJ6-L9B0CDPEmLBTSZMVq3x4UFUFCKs50MmVhQTNQ1VIrFxiTUKLZQrQEGV5sV-AHvH0iZPRyzlsX_kqhgACp7S_pyrLeILgAOfJXvB-GvU4T0uQ7dxmo3hB1bTF-rnJCXlUf_EGjfy_8JLwweRjqdURVYpTSO0qeFtkUhPrv0tf5Iv-d8FdZO2VbJAGLzAvjNXTACx_cYKhWCq6iNVlsWLsTM_s16xo_QTptklSjZCyXquptsfeXWJs3vvH9kKBppPS6LvAZ76wH85zHMiTYSG2BD1Ysd9DB-59jBj2ryTA9yrTH3DR8KinQQwop336ZlMThm38KBczrEuP-FZihoJtckAPzEZVUwvxaxjhTU7R7LaRmkcLJPwA2OjMPAY6EGh0d9Ks-4eQVmHgayp3QEeuN9Q0feSIkG2G8R1V-HqIMTEJjWgKUTVsw86JR8aHvvmuFlvRt77Q7rzKvtr2_f-ZOBJbsmChVW9lqN1rNLV0nlVDS6coZi-SgmDzM8rOqujZTLzpcIASEv7oF6OEvNRPa-8YFBrEeo6rnUNa5388L8qksG5bVgZVJ-QjAFlQldaOgd1Y5mFq5nbklZ0lRezzTT4Yj8g8x6u4lrGSaxvnyoO93DFqdz6oKc1V4XMqIAaxiU03WMdigHxg6sLYQvx7jT2vGlfqJ2vRCbXKtJnmpXIdh3hyqeXtNffiyDc-zPU9IJlZieRsKTF23JGUQL80nq82lURW4XKKcJpguai-Lg0CkFj_24J4VAHT1-VcbQxJRqefD_3SdZIARN456M2U2JqGuA6lceZYdivEjU2PETR-qCx06DZ_9ZC_YFt6x0weRuqpmGv97q-OFqV7K9a7XC4x9nbfIE_0YkE5caKjePOBEna82Umv6EXPI_HnVeyQt0XnYpW9x31AUkoIBHDYp933H_WmGrW1wcKYjN9CJnSom6jLdB2WYe10PYYe6VD1I4Y0nIwZBa7hkPFtBivSTXegHC0HtPWj_-wQXr8n3yzpfETp-ME5i0rapGDrmRTiGGPkeThKfUwZTvH7AQp0AcNlllFCsoD7d4vNAWChd-vrzs0io7ePXGcJ26VSvMqwka1-XUNLtmyw_uRWV9g3jrcG6QffC7IFmtnk49PpX8BSqJdOKp_BgMrxjWJVBJpedTc2pZUvGgjYHCxXx75aHP7VTiTXWL2lsSrsc6rmxqYvRIJ0zU88PycJZhxo10tmoO6ivRUPMavHCIujbsiirdJiCoE6_hvc7oE1nkRJX-ngmisdCRJT-9jgZ6Th6qZF4eBhCovseFhRJjHpMNVsUn_3Om6cCGDD2ShsG25EgwmNeHChnqPWokIeTum43ImuNNu-Q39zWNpVLhkUwQ6KBK_IApdfuFEheMMG_98-vpESSk7tPdz6JujkhwHVGj3l9N7JMgG7CkV7n77DkXmZweCt9K9hpWi23q_vpqKhcSauyycK8iFs6P0QjZIPCSo3E9yLhXOT7mfUqat20Fr0CNXOAMtj6vdVn_SZvhuSIswFyS1Ydxcr3uqMou_yxI-OiNHXaWBg9nyB8fRQMDbCDXO5lzzfzXXODbaxyUMNl13kPxoNgeiYUbaBhf4Z0uKKC8oXF8h5ob7zopBJ7a9YDnSD1oaysn3txFbOQebpTyTVubfz45IKNfuydZCRSejjkrRkV9NFJTz3P4QAyI0zzSXVFo3YkGRvrHEZn-_-mEfyKaAE1UtPiyFa2mFZADA6kiiISaNdVA3QjRdHXgqr0D_oPox35_jqMUM2aEOi4SG6hqVaWG_OiTYmWJvmeA.mhmVYknT_LxdTyuzrEJS9w"
}

In [ ]:
response = identify_user(**request)

In [48]:
response.text

'<HTML><HEAD>\n<TITLE>Access Denied</TITLE>\n</HEAD><BODY>\n<H1>Access Denied</H1>\n \nYou don\'t have permission to access "http&#58;&#47;&#47;id&#46;meijer&#46;com&#47;failover&#47;sitedown&#46;html&#63;" on this server.<P>\nReference&#32;&#35;18&#46;8f1c2117&#46;1744680170&#46;62b7f87d\n<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;8f1c2117&#46;1744680170&#46;62b7f87d</P>\n</BODY>\n</HTML>\n'